# Bedrock Module · Day 17 — Bedrock + LangGraph Integration

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~2.5 hrs

Module 1 built agents on LangGraph with Azure OpenAI. Barclays' platform is **Bedrock**. Today you do the migration that matters: **swap the model, keep the graph.** LangChain's `ChatBedrock` (from `langchain-aws`) is a drop-in chat model — same `.invoke` / `.stream` / `.bind_tools` / `.with_structured_output` surface as any LangChain model — so a well-built graph doesn't care whether Claude is served by Azure or Bedrock. That's this morning's **Strategy** pattern at framework scale.

You'll run a **real LangGraph `StateGraph`** whose model node is a `FakeChatBedrock` (same LangChain interface as `ChatBedrock`), then exercise the four things the migration must preserve: **streaming**, **tool-use**, **structured output**, and a **latency** comparison (Haiku vs Sonnet).

> **Keyless & offline.** LangGraph is real; the model is a `FakeChatBedrock` with the `ChatBedrock` interface (returns `AIMessage`, supports `bind_tools` / structured output). Swap it for `ChatBedrock(model_id=...)` = one line, shown in the closing table.

## 0. Setup — a LangChain-shaped Bedrock stand-in

`ChatBedrock` isn't installed (no boto3/langchain-aws here), so we build `FakeChatBedrock` with the **same interface**: `.invoke(messages) -> AIMessage`, plus `bind_tools` and `with_structured_output`. Everything downstream treats it exactly as it would the real class.

In [1]:
import time, re, json
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

class FakeChatBedrock:
    """Same surface as langchain_aws.ChatBedrock — swap the class, keep the code."""
    def __init__(self, model_id="claude-haiku", latency=0.03):
        self.model_id = model_id
        self.latency  = latency
        self._tools   = []
        self._schema  = None

    def invoke(self, messages):
        time.sleep(self.latency)                         # stands in for network + inference
        user = next((m.content for m in reversed(messages) if isinstance(m, HumanMessage)), "")
        # tool-use: if bound a tool and the ask needs it, return a tool_call (no content)
        if self._tools and "balance" in user.lower():
            acc = (re.search(r"ACC-\d+", user) or ["ACC-1001"])[0] if re.search(r"ACC-\d+", user) else "ACC-1001"
            return AIMessage(content="", tool_calls=[
                {"name": "get_balance", "args": {"account_id": acc}, "id": "call-1"}])
        # structured output: emit JSON matching the bound schema
        if self._schema:
            return AIMessage(content=json.dumps({"intent": "balance_enquiry", "urgency": "low"}))
        return AIMessage(content=f"[{self.model_id}] I can help with your Barclays account.")

    def bind_tools(self, tools):
        self._tools = tools; return self                 # returns self (chainable), like LangChain

    def with_structured_output(self, schema):
        self._schema = schema; return self

llm = FakeChatBedrock("claude-haiku")
print(llm.invoke([HumanMessage(content="hi")]).content)

[claude-haiku] I can help with your Barclays account.


☝ `FakeChatBedrock` returns a real `AIMessage` and exposes `bind_tools` / `with_structured_output` — the exact methods `ChatBedrock` has. Because the graph below depends only on this interface (Strategy!), the migration to real Bedrock is changing which object is assigned to `llm`. Nothing else moves.

## 1. Drop it into a real LangGraph `StateGraph`

Here's a genuine LangGraph agent: a state with a message list, one `model` node that calls `llm.invoke`, compiled into a runnable graph. The node calls **the interface**, so the model underneath is swappable — Azure yesterday, Bedrock today.

In [2]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]              # reducer appends new messages

def model_node(state: ChatState):
    reply = llm.invoke(state["messages"])                # <- the swappable Bedrock call
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("model", model_node)
builder.add_edge(START, "model")
builder.add_edge("model", END)
graph = builder.compile()

out = graph.invoke({"messages": [HumanMessage(content="What can you do?")]})
print(out["messages"][-1].content)

[claude-haiku] I can help with your Barclays account.


☝ A real compiled LangGraph produced that answer, and `model_node` never named Bedrock or Azure — it called `llm.invoke`. That single indirection is the entire migration story: your graph topology, state, and edges are **model-agnostic**. Pointing `llm` at `ChatBedrock` moves the whole app to Bedrock without touching the graph.

## 2. Streaming — token chunks for a responsive UI

Bedrock supports streaming; `ChatBedrock` exposes it as `.stream()` yielding chunks. Add a `stream` method to the stand-in and consume it like the real thing — the UX pattern from Day 7, now on the Bedrock path.

In [3]:
def _stream(self, messages):
    text = self.invoke(messages).content or "[tool call — nothing to stream]"
    for tok in text.split(" "):
        time.sleep(0.005); yield tok + " "
FakeChatBedrock.stream = _stream                          # ChatBedrock has .stream natively

print("streaming: ", end="")
for chunk in llm.stream([HumanMessage(content="summarise my options")]):
    print(chunk, end="", flush=True)
print()

streaming: [claude-haiku] I can help with your Barclays account. 


☝ Same content, delivered incrementally — the user sees words appear instead of waiting for the full response. With real `ChatBedrock` you'd iterate `.stream()` identically (or `graph.stream(...)` for node-level streaming). Streaming is a property of the model surface, so it survives the Azure→Bedrock swap for free.

## 3. Tool-use — bind a tool, execute, feed the result back

Agents *act* via tools. `bind_tools` tells the model what it may call; the model returns `tool_calls`; you execute them and pass a `ToolMessage` back for the final answer. This is the reason→act→observe loop (Day 13) on the LangChain/Bedrock surface.

In [4]:
ACCOUNTS = {"ACC-1001": 4200.0, "ACC-2002": 55000.0}
def get_balance(account_id: str) -> dict:
    return {"account_id": account_id, "balance": ACCOUNTS.get(account_id, 0.0)}

bound = FakeChatBedrock("claude-sonnet").bind_tools([get_balance])   # like ChatBedrock.bind_tools

msgs = [HumanMessage(content="what's the balance on ACC-2002?")]
ai = bound.invoke(msgs)                                   # model asks for a tool
print("tool_calls:", ai.tool_calls)

call = ai.tool_calls[0]
result = get_balance(**call["args"])                      # you execute it
msgs += [ai, ToolMessage(content=json.dumps(result), tool_call_id=call["id"])]
final = FakeChatBedrock("claude-sonnet").invoke(msgs)
print("observed:", result, "->", f"Balance is {result['balance']:.0f} GBP")

tool_calls: [{'name': 'get_balance', 'args': {'account_id': 'ACC-2002'}, 'id': 'call-1', 'type': 'tool_call'}]
observed: {'account_id': 'ACC-2002', 'balance': 55000.0} -> Balance is 55000 GBP


☝ The model returned a **`tool_call`** (name + args), your code ran `get_balance`, and the result went back as a `ToolMessage`. Real `ChatBedrock` + LangGraph's `ToolNode` automate the execute/feed-back step, but the shape is identical — Bedrock Claude does tool-use exactly like the Azure model your Module-1 agents used. Tools port unchanged.

## 4. Structured output — a typed object, not a string

For downstream code you want a **parsed object**, not prose. `with_structured_output(Schema)` makes the model emit data matching a Pydantic schema. Same method on `ChatBedrock`; you get a validated object out.

In [5]:
from pydantic import BaseModel

class Triage(BaseModel):
    intent: str
    urgency: str

structured = FakeChatBedrock("claude-sonnet").with_structured_output(Triage)
raw = structured.invoke([HumanMessage(content="I can't see my balance")])
parsed = Triage.model_validate_json(raw.content)          # real ChatBedrock returns the object directly
print("typed:", parsed, "| intent:", parsed.intent, "| urgency:", parsed.urgency)

typed: intent='balance_enquiry' urgency='low' | intent: balance_enquiry | urgency: low


☝ The model's answer came back as a **`Triage` object** with typed fields, ready to branch on — no brittle string parsing. (Real `ChatBedrock.with_structured_output` hands you the Pydantic instance directly; the stand-in emits JSON we validate, to stay dependency-light.) Structured output is how an agent's decision becomes reliable control flow — and it works the same on Bedrock.

## 5. Latency — Haiku vs Sonnet on Bedrock

Bedrock serves a model family; the migration is also a chance to pick the *right* size. Haiku is faster/cheaper, Sonnet stronger. Measure both on the same task and choose per use-case (support → Haiku, fraud → Sonnet, echoing Day-17 Python's factory).

In [6]:
def bench(model_id, latency, n=5):
    m = FakeChatBedrock(model_id, latency=latency)
    t0 = time.perf_counter()
    for _ in range(n):
        m.invoke([HumanMessage(content="triage this message")])
    return (time.perf_counter() - t0) / n * 1000          # avg ms/call

haiku_ms  = bench("claude-haiku",  latency=0.02)
sonnet_ms = bench("claude-sonnet", latency=0.06)          # heavier model = higher latency
print(f"Haiku : {haiku_ms:5.1f} ms/call")
print(f"Sonnet: {sonnet_ms:5.1f} ms/call   ({sonnet_ms/haiku_ms:.1f}x slower)")
print("rule of thumb: Haiku for high-volume FAQ, Sonnet for high-stakes reasoning")

Haiku :  25.1 ms/call
Sonnet:  64.8 ms/call   (2.6x slower)
rule of thumb: Haiku for high-volume FAQ, Sonnet for high-stakes reasoning


☝ Sonnet's latency is the price of its reasoning — so route by need, don't default everything to the biggest model. In a real benchmark you'd also compare **quality** on your eval suite (Day 15) and **cost per 1k tokens**. Bedrock makes the swap trivial (`model_id` string), so latency/quality/cost becomes a config choice, not a rewrite.

## 6. How this maps to real Bedrock + LangGraph

| You built | Real |
|---|---|
| `FakeChatBedrock` | `from langchain_aws import ChatBedrock` |
| `FakeChatBedrock("claude-haiku")` | `ChatBedrock(model_id="anthropic.claude-3-5-haiku-...")` |
| `.invoke / .stream` | same methods on `ChatBedrock` |
| `.bind_tools([...])` | `ChatBedrock.bind_tools(...)` + LangGraph `ToolNode` |
| `.with_structured_output(Schema)` | returns the Pydantic object directly |
| the `StateGraph` | **unchanged** — that's the point |

```python
# the ONE-LINE migration (needs langchain-aws + AWS access)
from langchain_aws import ChatBedrock
llm = ChatBedrock(model_id="anthropic.claude-3-5-sonnet-20241022-v2:0",
                  region_name="eu-west-2")
# every node above (model_node, bind_tools, structured, stream) works as-is
```

## 7. Recap — swap the model, keep the graph

| Capability | Preserved across Azure → Bedrock |
|---|---|
| **graph topology** | `StateGraph` untouched — model behind an interface |
| **streaming** | `.stream()` token chunks |
| **tool-use** | `bind_tools` → `tool_calls` → `ToolMessage` |
| **structured output** | `with_structured_output(Schema)` → typed object |
| **model choice** | Haiku vs Sonnet = a `model_id` string |

**Barclays lens:** the migration to Bedrock is a **one-line model swap**, not a rewrite — *if* your agent depends on the model interface, not a concrete SDK (Strategy, Day 14). Then you gain Bedrock's model catalogue, guardrails (Day 16), and AgentCore hosting (Day 14) without rebuilding the agent. That's the payoff of building on interfaces.
